# Week 6 · Your Week 5 work, reviewed, and how this week works

I read your Week 5 log (28 hours, Nov 10–17), the cells you added to every notebook, and the booklet sessions where you chose the kind of Python code you want. Three things change this week, all because of what you wrote. **Explanations** are longer and in plain words, in the format of your own rewrites of momentum and RMSProp (§D). **Words** like batch size, forward pass and upstream gradient are defined before they're used (§B). **Coding exercises** now come only from your booklet's "In Python" blocks, or from new blocks written the same way (§C). This notebook is short: read it, do one warm-up exercise, and start `01`.

**How to use this notebook**
- Run cells one at a time with **Shift + Enter**.
- Before running an example, **guess what it prints**. The real result is written after `# →` in the comments, so you can check yourself.
- **Your turn** cells are for you. Open **Show one answer** only after a real attempt.
- If things get weird, click **Restart** at the top and run from the top again.

**Run this cell first.** It loads NumPy, the picture helpers (`figures.py` in this folder) and the exercise checker.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from figures import *
import checks; checks.load(globals(), "00")      # defines check(n)

## M. The map: what's in this notebook and why

🖼 **Picture.** This week. `01` goes back over gradient descent and backpropagation slowly, as you asked; `02`–`03` are about training networks that work on new data; `04`–`05` are networks for images.

In [ ]:
fig_week6_map()

| Section | What | Why it's here |
|---|---|---|
| 0 | recap, and one booklet block from memory | the warm-up is now always a booklet "In Python" block |
| A | your Week 5 log and notes | what you asked for, and where this week answers it |
| B | the words of deep learning | "You need to be more clear about the terminology": every word, with a picture |
| C | coding from now on | "dont expect me to write `gd_steps` from scratch": the new rule |
| D | explanations from now on | "explanations need to be much better": the format, copied from your own rewrites |

## 0. Recap first · 10 min, closed book

**Quick check:** In one sentence each: what does the **forward pass** compute, and what does the **backward pass** compute?

<details><summary>Answer</summary>

Forward: from the input, layer by layer, to the prediction and then the loss (one number). Backward: from the loss back to the input, the gradient of the loss with respect to every weight. (§B below draws both.) [Week 5 · 03 §E ↗](../week5/03_backprop.ipynb)

</details>

**Quick check:** For softmax followed by cross-entropy, $\partial L / \partial z = $ ?

<details><summary>Answer</summary>

$p - y$ (with $y$ one-hot): the predicted probabilities minus the true ones. [Week 5 · 02 §7 ↗](../week5/02_neural_networks.ipynb)

</details>

**Quick check:** Momentum's update in words?

<details><summary>Answer</summary>

Step along a running, fading sum of recent gradients instead of the latest one: steady directions add up (up to 10× for $\beta = 0.9$), zigzags cancel. (Your own sentence from 05 §3.) [Week 5 · 05 §3 ↗](../week5/05_optimizers.ipynb)

</details>

**Quick check:** The five lines inside every PyTorch training loop, in order?

<details><summary>Answer</summary>

`logits = model(xb)`, `loss = loss_fn(logits, yb)`, `opt.zero_grad()`, `loss.backward()`, `opt.step()`. [Week 5 · 06 §5 ↗](../week5/06_pytorch_basics.ipynb)

</details>

**Quick check:** `model.eval()` and `torch.no_grad()`: which one changes how layers behave, and which one stops gradient recording?

<details><summary>Answer</summary>

`eval()` changes layers (dropout off, batch norm uses stored statistics); `no_grad()` stops recording. Your "two switches". [Week 5 · 06 §7 ↗](../week5/06_pytorch_basics.ipynb)

</details>

**Code from memory: now always a booklet block.** Each notebook's warm-up is one of your booklet's "In Python" blocks, written closed book, then checked. This one is booklet **2.5** (logistic regression): the gradient-descent loop, as a function.

**Exercise 1 · Booklet 2.5: logistic regression by gradient descent.** Start from $w = 0$, $b = 0$. Repeat `steps` times: $\hat p_i = \sigma(w^\top x_i + b)$; $w \leftarrow w - \eta\, \frac{1}{n}\sum_i (\hat p_i - y_i)\, x_i$; $b \leftarrow b - \eta\, \frac{1}{n}\sum_i (\hat p_i - y_i)$. Return `(w, b)`. `X` is `(n, d)`, `y` is `(n,)` of 0/1, `lr` is $\eta$.

**Example**: what goes in, and what should come out:
```python
X = np.array([[0.0], [1.0], [2.0], [3.0]]); y = np.array([0.0, 0.0, 1.0, 1.0])
w, b = logistic_gd(X, y, 0.5, 1)    # one step from w = 0, b = 0: every p = 0.5
w, b                               # → (array([0.25]), np.float64(0.0))   w = 0 − 0.5 × (1/4)(0.5·0 + 0.5·1 − 0.5·2 − 0.5·3) = 0.25
```

In [ ]:
def logistic_gd(X, y, lr, steps):
    pass  # your code

In [ ]:
check(1)

## A. Your Week 5, reviewed

You did the notebooks in your own order (`02` first, "because I wanted to study something new"), finished all of them, and wrote long explanation cells in `04`, `05` and `06`. Here's what you asked for, and where this week answers each request.

| You wrote | What changes |
|---|---|
| "You need to be more clear about the terminology or the jargon... batch size, forward pass, upstream, down stream" | §B below defines every word with a picture. From now on a new word is explained the moment it first appears. |
| "I can't wrap my head around gradient descent cleanly enough this needs revision" | `01` §1–3 rebuilds gradient descent from one weight, with pictures and numbers. |
| "Visualizations would be nice where matrices start to come in... §5 for example" | `01` §4 draws every matrix of the backprop algorithm, forward and backward, with its shape. |
| "didnt really understand §7" (vanishing gradients) | `01` §5 explains it again, slowly, as a multiplication of many small numbers. |
| "dont expect me to write `gd_steps`... from scratch"; "I am skeptical as to whether the exam is going to ask me to build it from scratch" | Coding exercises now come only from your booklet's "In Python" blocks (§C). |
| "Your explanations are also too concise... You lack explanation on the proposition to proof format" | The format in §D, copied from your own rewrite of momentum in `05`. |
| "i need smaller stepping stones for pytorch" | `05` builds the CNN one layer at a time and checks the shape after every layer before training anything. |

**Your notes, checked**

| Where | What you wrote | Verdict |
|---|---|---|
| [01 §4 ↗](../week5/01_formula_to_numpy.ipynb) | (in Korean) "treat the trailing 2 in (2, 2) as if it weren't there; think of [2, 2] as one element" | Correct, and a good way to think about broadcasting: an axis both arrays share just comes along for the ride. |
| [01 §6 ↗](../week5/01_formula_to_numpy.ipynb) | `A.T @ B` = column × column dot products, `A @ B.T` = row × row, with your own pictures | Correct. These two are exactly the two backprop products (`H.T @ G` and `G @ W.T`). |
| [02 §4 ↗](../week5/02_neural_networks.ipynb) | "non-linearity is essential... if not the model would collapse into a simple linear model... captures interaction effects" | Correct: the proposition in one sentence. |
| [02 §7 ↗](../week5/02_neural_networks.ipynb) | your own example: $z = (2, 1, 0)$, $y = 2$: $p = (0.665, 0.245, 0.090)$ | Correct; the loss is $-\log 0.090 = 2.41$, large because the true class got the smallest score. |
| [03 §3 ↗](../week5/03_backprop.ipynb) | (in Korean) "don't forget the gradient at the pre-activation step" | That's exactly the step most people miss: $\partial L / \partial z = \partial L / \partial h \odot \sigma'(z)$ before going further back. |
| [04 §1 ↗](../week5/04_micrograd.ipynb) | the call-by-call trace of `visit("o")` | Correct, and it's the right way to understand any recursion: write the calls out. |
| [05 §3–4 ↗](../week5/05_optimizers.ipynb) | your rewrites of momentum and RMSProp: the problem, the idea, the formula piece by piece, what the proposition says, a $t = 3$ example, the proof in plain words, the remarks, one sentence | Correct, and better than mine. It's the template for this week (§D). |
| [06 §6–7 ↗](../week5/06_pytorch_basics.ipynb) | a `Dataset` is a recipe book, not a fridge; `eval()` and `no_grad()` are two separate switches | Both correct, and the analogies are worth keeping. |

**Scores.** `07` and `08` are ticked but their scores aren't in the log. Please add them (paper practice out of 100, quiz A /10, B /3, C /5) with the misses by topic, so `06` and `07` this week can target them.

## B. The words of deep learning

Every term you'll meet this week, in plain words. The picture after the table shows most of them in one training step.

| Word | In plain words | Example |
|---|---|---|
| sample | one data point: one image, one row of the table | one 8×8 digit image = 64 numbers |
| feature | one number describing a sample: one column | the brightness of pixel 10 |
| batch | a few samples processed together | 4 images stacked into a `(4, 64)` array |
| batch size | how many samples are in one batch | `DataLoader(..., batch_size=64)` |
| forward pass | running a batch through the network, input to prediction to loss | `logits = model(xb)`, then `loss = loss_fn(logits, yb)` |
| loss | one number saying how wrong the predictions on this batch are | mean cross-entropy, e.g. 0.6 |
| gradient | for every weight: how much the loss changes if that weight moves a little, and in which direction | $\partial L / \partial w = -0.28$: raising $w$ lowers the loss |
| backward pass | computing all the gradients, from the loss back towards the input | `loss.backward()` |
| upstream gradient | the gradient arriving at a box **from the loss side** | at the σ box below: $\partial L / \partial h$ |
| local gradient | the box's own derivative, known from the box alone | $\sigma'(z) = h(1 - h)$ |
| downstream gradient | upstream × local: what the box **sends on** towards the input | $\partial L / \partial z$ |
| learning rate $\eta$ | how big a step to take against the gradient | `lr=0.01` |
| step (iteration) | one weight update, from one batch | `opt.step()` |
| epoch | one pass over all the training samples: (number of samples ÷ batch size) steps | 1347 digits, batch 64: 22 steps |
| parameter | a number the network **learns** | the weights and biases |
| hyperparameter | a number **you choose** before training | learning rate, batch size, number of layers |

🖼 **Picture.** One training step. The batch goes **forward** (blue) through the model to the loss; the loss goes **backward** (red) to a gradient for every weight; one **update** uses them. That's one step.

In [ ]:
fig_training_step()

🖼 **Picture.** Steps and epochs. With 10 samples and batch size 4, one epoch is 3 steps; the next epoch shuffles and starts again.

In [ ]:
fig_epoch_steps()

🖼 **Picture.** Upstream and downstream, on a tiny chain $x \to z = wx \to h = \sigma(z) \to L = -\log h$. At every box, the gradient that arrives from the right (upstream) is multiplied by the box's own derivative (local) and sent on to the left (downstream).

In [ ]:
fig_upstream_downstream()

Read the σ box: the gradient arriving from the loss is $\partial L / \partial h = -1/h = -1.222$ (upstream). The box's own derivative is $\sigma'(z) = h(1 - h) = 0.818 \times 0.182 = 0.149$ (local). It sends $-1.222 \times 0.149 = -0.182$ on to the left (downstream). Every box does this one multiplication; backprop is that, repeated from right to left.

## C. Coding from now on

**The rule.** Every coding exercise is either one of your booklet's "In Python" blocks, written from memory, or a new block written at the same level for a new topic. That level is the one you chose in the booklet session: write the chapter's **core loop or formula** by hand in NumPy; take everything underneath (trees, eigenvectors, layers, autograd, optimizers) from scikit-learn, SciPy or PyTorch; and add **one line that checks** your result against the package.

**Why.** It's what a handwritten exam answer looks like: short, every line maps to a formula, and library calls are allowed. Building whole engines from scratch (`04` micrograd) was useful once, but it costs hours that the exam won't reward.

| Notebook | Its coding exercises come from |
|---|---|
| `00` | booklet 2.5 (logistic regression) |
| `01` | booklet 1.8 (linear regression by gradient descent) and 13.7 (backprop: the backward function) |
| `02` | booklet 1.8 (ridge, closed form), then the new regularization block (weight decay, dropout, early stopping) |
| `03` | booklet 14 (Adam), then the new stability block (batch norm, layer norm, clipping) |
| `04` | booklet 9.4 (the EM E-step), then the new convolution block (output size, convolution by hand, pooling) |
| `05` | booklet 15 (the training loop), then the new CNN block |
| `06`, `07` | booklet 5, 3 and 15, and this week's new blocks |

The new blocks are written the way your booklet's are, so each can become the "In Python" section of a new booklet chapter (Regularization, Training stability, Convolutional networks) when you write your review notes.

**How to practise each exercise.** (1) Read the description and the example. (2) Write the function on paper. (3) Type it and run the check. (4) If you're stuck after 15 minutes, read the booklet block, close it, and write it again from a blank cell.

## D. Explanations from now on

Your rewrite of momentum in `05` is the clearest explanation in the whole pack. Its structure, which every explanation this week follows:

- **The problem**: what goes wrong without the idea, with concrete numbers ("across: gradients around 10; along: around 0.1").
- **The idea**: one plain paragraph, often an everyday picture.
- **The formula, piece by piece**: each symbol explained in words right after the formula ("new velocity = 0.9 × old velocity + today's gradient").
- **A picture** where there's anything to see, especially when matrices get big.
- **A tiny example by hand**, every number written out ("example with $t = 3$").
- For a proposition: **what it says, in words** → **a small example** → **the proof in plain words** → the formal proof → **the remarks**, each one explained → **in one sentence**.
- **New words** defined where they first appear.

Here's the format on something you already know, so you can see it once before it's used on new material.

**Example: why does k-means move each centre to the mean of its points?**

**The problem.** A cluster has the points 1, 2 and 6 (on a number line). Where should its centre go so that the total squared distance to the points is as small as possible? At 2 the total is $1 + 0 + 16 = 17$; at 4 it's $9 + 4 + 4 = 17$; somewhere in between it's smaller.

**Proposition.** For numbers $x_1, \dots, x_n$, the value $c$ that minimizes $J(c) = \sum_i (x_i - c)^2$ is their mean $\bar x$.

**What it says, in words.** If you have to pick one spot that's "closest on average", in the squared-distance sense, pick the average. **Small example:** for 1, 2, 6 the mean is 3, and $J(3) = 4 + 1 + 9 = 14$, less than the 17 at 2 or at 4.

**The proof in plain words.** $J$ is a parabola in $c$ (a sum of squares), so its lowest point is where its slope is zero. The slope is $-2\sum_i (x_i - c)$; it's zero when $\sum_i x_i = n c$, i.e. $c = \bar x$.

$$
J'(c) = \sum_{i=1}^{n} -2(x_i - c) = -2\Big(\sum_{i=1}^{n} x_i - n c\Big) = 0 \iff c = \frac{1}{n}\sum_{i=1}^{n} x_i = \bar x. \qquad \blacksquare
$$

**Remark.** It's a minimum, not a maximum, because $J''(c) = 2n > 0$: the parabola opens upwards. That's why k-means' update step can never increase $J$ (your Lloyd's algorithm card, Week 4).

**In one sentence:** the mean is the point with the smallest total squared distance, because that's where the parabola's slope is zero.

---
Solutions: `solutions/00_solutions.py`. Next: `01_gd_and_backprop_slowly.ipynb`.